# Parquet Basics — 08: Compression Codecs Deep Dive

## What you will learn
Choosing the right compression codec saves storage costs and improves performance.
The "best" codec depends on your data, hardware, and access patterns.

In this notebook:
1. How compression works inside Parquet (page-level)
2. Full benchmark: snappy vs zstd vs gzip vs lz4 vs none vs brotli
3. CPU vs I/O trade-off — when more compression is slower
4. zstd compression levels — fine-tuning ratio vs speed
5. Data type impact — which columns compress best
6. Production recommendations by workload type


In [1]:
import os, time, pathlib, shutil, random, datetime, subprocess
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/parquet_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('parquet-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | DATA_DIR: {DATA_DIR}")

random.seed(42)
N = 200_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("discount",    DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("is_returned", BooleanType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2023, 1, 1)
rows = []
for i in range(N):
    qty  = random.randint(1, 20)
    price = round(random.uniform(5, 2000), 2)
    disc  = round(random.uniform(0, 0.4), 3)
    rev   = round(qty * price * (1 - disc), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 365))
    rows.append((i+1, random.randint(1,50000),
                 f"Product_{random.randint(1,500)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, disc, rev,
                 random.choice(STATUSES), random.random() < 0.05, d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset: {N:,} rows | {len(schema)} columns")
df.printSchema()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 21:20:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | DATA_DIR: /workspace/data/parquet_basics


26/04/10 21:20:16 WARN TaskSetManager: Stage 0 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 21:20:18 WARN TaskSetManager: Stage 1 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.


Dataset: 200,000 rows | 12 columns
root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- country: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- status: string (nullable = true)
 |-- is_returned: boolean (nullable = true)
 |-- order_date: date (nullable = true)



## Step 1 — How Compression Works in Parquet

Parquet compresses at the **page level** (default 1 MB pages) within each column chunk.
Each page is compressed independently using the chosen codec.

```
Column Chunk (revenue):
  Page 0: [999.99, 49.99, 349.99, ...]  → compressed with codec → stored bytes
  Page 1: [129.99, 599.99, ...]         → compressed with codec → stored bytes
  ...

Dictionary page (category): ["Electronics","Books",...] → compressed once
Data pages (category):       [0, 1, 2, 0, 1, ...]      → indices, compress very well
```

This is why low-cardinality string columns compress so much better than floats —
the dictionary page holds all unique values, data pages hold tiny integer indices.


In [2]:
import glob as glib

codecs_to_test = ["snappy", "zstd", "gzip", "lz4", "none"]
try:
    # brotli available in some pyarrow versions
    import pyarrow.parquet as pq
    codecs_to_test.append("brotli")
except:
    pass

OUT = f"{DATA_DIR}/codecs"
results = {}

print(f"Testing {len(codecs_to_test)} compression codecs on {N:,} rows...")
print()

for codec in codecs_to_test:
    path = f"{OUT}/{codec}"
    try:
        # Write
        t0 = time.time()
        df.write.mode("overwrite").option("compression", codec).parquet(path)
        t_write = time.time() - t0

        # Size
        files   = glib.glob(f"{path}/*.parquet")
        size_mb = sum(pathlib.Path(f).stat().st_size for f in files) / 1024/1024

        # Read + full scan
        times = []
        for _ in range(3):
            t0 = time.time()
            spark.read.parquet(path).agg(F.sum("revenue"), F.count("*")).collect()
            times.append(time.time() - t0)
        t_read = sorted(times)[1]

        results[codec] = {"write": t_write, "size": size_mb, "read": t_read}
        print(f"  {codec:<8} write={t_write:.2f}s  size={size_mb:.1f}MB  read={t_read:.3f}s")
    except Exception as e:
        print(f"  {codec:<8} ERROR: {e}")

Testing 6 compression codecs on 200,000 rows...



26/04/10 21:20:19 WARN TaskSetManager: Stage 4 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

  snappy   write=2.11s  size=4.6MB  read=0.521s


26/04/10 21:20:23 WARN TaskSetManager: Stage 17 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.


  zstd     write=0.92s  size=3.1MB  read=0.412s


...

26/04/10 21:20:29 WARN TaskSetManager: Stage 56 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.


  none     write=0.88s  size=7.2MB  read=0.329s


26/04/10 21:20:31 WARN TaskSetManager: Stage 69 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 21:20:31 WARN TaskSetManager: Lost task 3.0 in stage 69.0 (TID 122) (172.18.0.5 executor 0): org.apache.parquet.hadoop.BadConfigurationException: Class org.apache.hadoop.io.compress.BrotliCodec was not found
	at org.apache.parquet.hadoop.CodecFactory.getCodec(CodecFactory.java:309)
	at org.apache.parquet.hadoop.CodecFactory.createCompressor(CodecFactory.java:272)
	at org.apache.parquet.hadoop.CodecFactory.getCompressor(CodecFactory.java:255)
	at org.apache.parquet.hadoop.ParquetRecordWriter.<init>(ParquetRecordWriter.java:210)
	at org.apache.parquet.hadoop.ParquetOutputFormat.getRecordWriter(ParquetOutputFormat.java:571)
	at org.apache.parquet.hadoop.ParquetOutputFormat.getRecordWriter(ParquetOutputFormat.java:467)
	at org.apache.parquet.hadoop.ParquetOutputFormat.getRecordWriter(ParquetOutputFormat.java:456)
	at org.apache.spark.sql.exe

  brotli   ERROR: An error occurred while calling o312.parquet.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 69.0 failed 4 times, most recent failure: Lost task 0.3 in stage 69.0 (TID 133) (172.18.0.4 executor 1): org.apache.parquet.hadoop.BadConfigurationException: Class org.apache.hadoop.io.compress.BrotliCodec was not found
	at org.apache.parquet.hadoop.CodecFactory.getCodec(CodecFactory.java:309)
	at org.apache.parquet.hadoop.CodecFactory.createCompressor(CodecFactory.java:272)
	at org.apache.parquet.hadoop.CodecFactory.getCompressor(CodecFactory.java:255)
	at org.apache.parquet.hadoop.ParquetRecordWriter.<init>(ParquetRecordWriter.java:210)
	at org.apache.parquet.hadoop.ParquetOutputFormat.getRecordWriter(ParquetOutputFormat.java:571)
	at org.apache.parquet.hadoop.ParquetOutputFormat.getRecordWriter(ParquetOutputFormat.java:467)
	at org.apache.parquet.hadoop.ParquetOutputFormat.getRecordWriter(ParquetOutputFormat.java:456)
	at org.apache.spa

26/04/10 21:20:32 ERROR TaskSetManager: Task 0 in stage 69.0 failed 4 times; aborting job
26/04/10 21:20:32 ERROR FileFormatWriter: Aborting job 7ca29c2e-cf76-4a84-9b2f-8928ef40eb06.
org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 69.0 failed 4 times, most recent failure: Lost task 0.3 in stage 69.0 (TID 133) (172.18.0.4 executor 1): org.apache.parquet.hadoop.BadConfigurationException: Class org.apache.hadoop.io.compress.BrotliCodec was not found
	at org.apache.parquet.hadoop.CodecFactory.getCodec(CodecFactory.java:309)
	at org.apache.parquet.hadoop.CodecFactory.createCompressor(CodecFactory.java:272)
	at org.apache.parquet.hadoop.CodecFactory.getCompressor(CodecFactory.java:255)
	at org.apache.parquet.hadoop.ParquetRecordWriter.<init>(ParquetRecordWriter.java:210)
	at org.apache.parquet.hadoop.ParquetOutputFormat.getRecordWriter(ParquetOutputFormat.java:571)
	at org.apache.parquet.hadoop.ParquetOutputFormat.getRecordWriter(ParquetOutputFormat.java:46

In [3]:
# Summary table
base_size = results["none"]["size"]
base_write = results["none"]["write"]

print(f"\n{'Codec':<10} {'Write':>8} {'vs none':>8} {'Size MB':>10} {'Compression':>12} {'Read':>8}")
print("-" * 62)
for codec, r in results.items():
    write_ratio = f"{r['write']/base_write:.2f}x"
    compress_ratio = f"{base_size/r['size']:.2f}x"
    print(f"  {codec:<8} {r['write']:>6.2f}s {write_ratio:>8} "
          f"{r['size']:>8.1f} MB {compress_ratio:>12} {r['read']:>6.3f}s")

print()
if "zstd" in results and "snappy" in results:
    zstd_vs_snappy = results["snappy"]["size"] / results["zstd"]["size"]
    print(f"zstd vs snappy: {zstd_vs_snappy:.2f}x smaller, "
          f"{results['zstd']['write']/results['snappy']['write']:.2f}x slower write")


Codec         Write  vs none    Size MB  Compression     Read
--------------------------------------------------------------
  snappy     2.11s    2.40x      4.6 MB        1.55x  0.521s
  zstd       0.92s    1.05x      3.1 MB        2.30x  0.412s
  gzip       1.02s    1.16x      3.2 MB        2.23x  0.371s
  lz4        0.88s    1.00x      4.6 MB        1.58x  0.365s
  none       0.88s    1.00x      7.2 MB        1.00x  0.329s

zstd vs snappy: 1.48x smaller, 0.44x slower write


## Step 2 — zstd Compression Levels

zstd supports levels 1-22:
- Level 1: fast, similar to snappy
- Level 3: default — good balance (used by Spark's `zstd` option)
- Level 9: high compression, slower
- Level 19+: very high compression, much slower — rarely worth it


In [4]:
zstd_path = f"{DATA_DIR}/zstd_levels"
zstd_results = {}

for level in [1, 3, 6, 9, 15]:
    path = f"{zstd_path}/level_{level}"
    t0 = time.time()
    df.write.mode("overwrite") \
            .option("compression", "zstd") \
            .option("parquet.compression.codec.zstd.level", str(level)) \
            .parquet(path)
    t_write = time.time() - t0

    files = glib.glob(f"{path}/*.parquet")
    size_mb = sum(pathlib.Path(f).stat().st_size for f in files) / 1024/1024
    zstd_results[level] = {"write": t_write, "size": size_mb}
    print(f"  zstd level {level:>2}: {size_mb:.1f} MB  write={t_write:.2f}s")

base = zstd_results[1]["size"]
print()
print(f"  Level 1  → baseline (fastest zstd)")
for level, r in zstd_results.items():
    if level > 1:
        savings = (base - r["size"])/base * 100
        print(f"  Level {level:>2} → {savings:.0f}% smaller than level 1, "
              f"{r['write']/zstd_results[1]['write']:.1f}x slower write")

26/04/10 21:20:32 WARN TaskSetManager: Stage 70 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.


  zstd level  1: 3.5 MB  write=0.84s


26/04/10 21:20:33 WARN TaskSetManager: Stage 71 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.


  zstd level  3: 3.1 MB  write=0.79s


26/04/10 21:20:34 WARN TaskSetManager: Stage 72 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.


  zstd level  6: 3.0 MB  write=0.85s


26/04/10 21:20:34 WARN TaskSetManager: Stage 73 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.


  zstd level  9: 2.9 MB  write=0.92s


26/04/10 21:20:35 WARN TaskSetManager: Stage 74 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

  zstd level 15: 3.0 MB  write=1.67s

  Level 1  → baseline (fastest zstd)
  Level  3 → 10% smaller than level 1, 0.9x slower write
  Level  6 → 14% smaller than level 1, 1.0x slower write
  Level  9 → 15% smaller than level 1, 1.1x slower write
  Level 15 → 14% smaller than level 1, 2.0x slower write


## Step 3 — Compression by Data Type

Different data types compress very differently:
- **Low-cardinality strings** (category, country): excellent — dict encoding + compression
- **High-cardinality strings** (emails, UUIDs): poor — no dictionary benefit
- **Doubles/floats**: poor — near-random bit patterns
- **Integers with patterns**: good — delta encoding then compression
- **Booleans**: excellent — RLE + compression on bit-packed values


In [5]:
type_df = spark.createDataFrame([
    (i,
     f"cat_{i % 6}",              # low cardinality string
     f"user_{i}@email.com",       # high cardinality string
     float(i) * 3.14159,          # float (pseudo-random pattern)
     i % 1000,                    # integer with pattern
     i,                           # sequential integer
     i % 2 == 0,                  # boolean
    ) for i in range(200_000)
], ["id","category","email","amount","code","seq_id","is_active"])

type_path = f"{DATA_DIR}/type_compression"
for codec in ["snappy", "zstd", "none"]:
    path = f"{type_path}/{codec}"
    type_df.write.mode("overwrite").option("compression",codec).parquet(path)

# Measure per-column compression via pyarrow
try:
    import pyarrow.parquet as pq
    for codec in ["none", "snappy", "zstd"]:
        path = f"{type_path}/{codec}"
        files = glib.glob(f"{path}/*.parquet")
        if files:
            meta = pq.ParquetFile(files[0]).metadata
            rg   = meta.row_group(0)
            print(f"\nCodec: {codec}")
            for ci in range(min(7, meta.num_columns)):
                col = rg.column(ci)
                kb  = col.total_compressed_size / 1024 if hasattr(col, 'total_compressed_size') else col.total_byte_size / 1024
                print(f"  {col.path_in_schema:<20} {kb:>8.1f} KB")
except ImportError:
    # Fallback: measure file sizes per column via schema
    print("Column-level analysis (overall file sizes):")
    for codec in ["none","snappy","zstd"]:
        path = f"{type_path}/{codec}"
        files = glib.glob(f"{path}/*.parquet")
        mb = sum(pathlib.Path(f).stat().st_size for f in files) / 1024/1024
        print(f"  {codec:<8}: {mb:.1f} MB")

26/04/10 21:20:44 WARN TaskSetManager: Stage 75 contains a task of very large size (2595 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 21:20:45 WARN TaskSetManager: Stage 76 contains a task of very large size (2595 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 21:20:46 WARN TaskSetManager: Stage 77 contains a task of very large size (2595 KiB). The maximum recommended task size is 1000 KiB.



Codec: none
  id                      392.1 KB
  category                 18.7 KB
  email                  1165.3 KB
  amount                  392.1 KB
  code                     69.3 KB
  seq_id                  392.1 KB
  is_active                 6.2 KB

Codec: snappy
  id                      197.0 KB
  category                  1.1 KB
  email                   243.9 KB
  amount                  382.9 KB
  code                     11.6 KB
  seq_id                  197.0 KB
  is_active                 0.4 KB

Codec: zstd
  id                       67.6 KB
  category                  0.3 KB
  email                    27.6 KB
  amount                  252.2 KB
  code                      6.3 KB
  seq_id                   67.6 KB
  is_active                 0.2 KB


## Summary: Compression Codec Selection Guide

| Workload | Recommended | Reason |
|---|---|---|
| Hot data (frequent reads) | `snappy` | Fast decompression, good ratio |
| Cold/archival storage | `zstd` level 6+ | Best ratio, acceptable speed |
| Temp/intermediate files | `lz4` | Fastest, minimal CPU |
| Maximum compression | `zstd` level 9-15 | Best ratio when storage costs dominate |
| CPU-bound workloads | `none` | Zero decompression overhead |

### Quick decision tree
```
Is storage cost the primary concern?
  YES → zstd (level 3-6)
  NO  ↓
Is this a frequently queried hot table?
  YES → snappy (fast decompression)
  NO  ↓
Is this a temporary/intermediate file?
  YES → lz4 (fastest)
  NO  → snappy (safe default)
```

```python
# Storage-optimized
.option("compression", "zstd")
.option("parquet.compression.codec.zstd.level", "6")

# Performance-optimized
.option("compression", "snappy")   # default

# Maximum speed
.option("compression", "lz4")
```
